In [2]:
import pandas as pd


try:
    df_link = pd.read_csv('cast_movie.csv')
    df_movies = pd.read_csv('movies.csv')
    df_cast = pd.read_csv('cast.csv')
    print("Đọc thành công các file dữ liệu!")
except Exception as e:
    print(f"Lỗi đọc file: {e}")

# --- BƯỚC 1: KIỂM TRA VÀ XÓA TRÙNG LẶP CẶP KHÓA HỢP THÀNH ---
print(f"Kích thước file liên kết gốc: {df_link.shape}")


trung_lap_cap = df_link[df_link.duplicated(subset=['movie_id', 'cast_id', 'cast_role'], keep=False)]
print(f"Số lượng dòng bị trùng lặp cả cặp (movie_id, cast_id, cast_role): {len(trung_lap_cap)}")

# Tiến hành xóa trùng lặp, giữ lại dòng đầu tiên
df_link = df_link.drop_duplicates(subset=['movie_id', 'cast_id', 'cast_role'], keep='first')
print(f"Kích thước sau khi xóa trùng lặp cặp: {df_link.shape}")


# --- BƯỚC 2: KIỂM TRA TÍNH TOÀN VẸN THAM CHIẾU (FOREIGN KEY) ---
# Kiểm tra xem có movie_id nào trong file liên kết mà KHÔNG tồn tại bên file movies không
movie_mo_coi = df_link[~df_link['movie_id'].isin(df_movies['movie_id'])]
print(f"Số dòng chứa 'movie_id' lạ (không có trong bảng Movies): {len(movie_mo_coi)}")

# Kiểm tra xem có cast_id nào trong file liên kết mà KHÔNG tồn tại bên file cast không
cast_mo_coi = df_link[~df_link['cast_id'].isin(df_cast['cast_id'])]
print(f"Số dòng chứa 'cast_id' lạ (không có trong bảng Cast): {len(cast_mo_coi)}")

# 1. Tìm các bộ phim KHÔNG CÓ diễn viên nào đóng trong bảng liên kết
movies_khong_cast = df_movies[~df_movies['movie_id'].isin(df_link['movie_id'])]
print(f"Số lượng phim bị 'trống' diễn viên: {len(movies_khong_cast)}")

# 2. Tìm các diễn viên KHÔNG THAM GIA bộ phim nào trong bảng liên kết
cast_khong_movie = df_cast[~df_cast['cast_id'].isin(df_link['cast_id'])]
print(f"Số lượng diễn viên không đóng phim nào: {len(cast_khong_movie)}")


# BIỆN PHÁP XỬ LÝ (Tùy chọn): Lọc bỏ các dòng "mồ côi" này để tránh lỗi Foreign Key khi nạp SQL
df_link = df_link[df_link['movie_id'].isin(df_movies['movie_id'])]
df_link = df_link[df_link['cast_id'].isin(df_cast['cast_id'])]
print(f"Kích thước file liên kết sau khi làm sạch hoàn toàn: {df_link.shape}")
# 1. Chỉ giữ lại những phim thực sự xuất hiện trong bảng liên kết cast_movie
df_movies_cleaned = df_movies[df_movies['movie_id'].isin(df_link['movie_id'])]
# 2. Chỉ giữ lại những diễn viên thực sự xuất hiện trong bảng liên kết cast_movie
df_cast_cleaned = df_cast[df_cast['cast_id'].isin(df_link['cast_id'])]
# 3. Xuất ra các file sạch cuối cùng để nạp vào SQL
df_movies_cleaned.to_csv("movies_final.csv", index=False)
df_cast_cleaned.to_csv("casts_final.csv", index=False)
# Đổi tên cột id của cast nếu cần, ví dụ: df_cast_cleaned.to_csv("cast_final.csv", index=False)

print(f"Đã làm sạch! Số phim còn lại: {len(df_movies_cleaned)}, Số diễn viên còn lại: {len(df_cast_cleaned)}")

# Lưu lại file liên kết sạch
df_link.to_csv("movie_cast_cleaned.csv", index=False)
print("Đã xuất file sạch 'movie_cast_cleaned.csv' sẵn sàng nạp SQL!")


Đọc thành công các file dữ liệu!
Kích thước file liên kết gốc: (244152, 3)
Số lượng dòng bị trùng lặp cả cặp (movie_id, cast_id, cast_role): 0
Kích thước sau khi xóa trùng lặp cặp: (244152, 3)
Số dòng chứa 'movie_id' lạ (không có trong bảng Movies): 267
Số dòng chứa 'cast_id' lạ (không có trong bảng Cast): 0
Số lượng phim bị 'trống' diễn viên: 380
Số lượng diễn viên không đóng phim nào: 18
Kích thước file liên kết sau khi làm sạch hoàn toàn: (243885, 3)
Đã làm sạch! Số phim còn lại: 4868, Số diễn viên còn lại: 114853
Đã xuất file sạch 'movie_cast_cleaned.csv' sẵn sàng nạp SQL!


In [7]:
import pandas as pd


try:
    df_link = pd.read_csv('movie_cast.csv')
    df_movies = pd.read_csv('movies.csv')       # Đọc lại file movies sạch để đối chiếu
    df_cast = pd.read_csv('casts.csv')
    print("Đọc thành công các file dữ liệu!")
except Exception as e:
    print(f"Lỗi đọc file: {e}")

# --- BƯỚC 1: KIỂM TRA VÀ XÓA TRÙNG LẶP CẶP KHÓA HỢP THÀNH ---
print(f"Kích thước file liên kết gốc: {df_link.shape}")

# Kiểm tra trùng lặp dựa trên CẢ HAI cột (Composite Key)
trung_lap_cap = df_link[df_link.duplicated(subset=['movie_id', 'cast_id', 'cast_role'], keep=False)]
print(f"Số lượng dòng bị trùng lặp cả cặp (movie_id, cast_id): {len(trung_lap_cap)}")

# --- BƯỚC 2: KIỂM TRA TÍNH TOÀN VẸN THAM CHIẾU (FOREIGN KEY) ---
# Kiểm tra xem có movie_id nào trong file liên kết mà KHÔNG tồn tại bên file movies không
movie_mo_coi = df_link[~df_link['movie_id'].isin(df_movies['movie_id'])]
print(f"Số dòng chứa 'movie_id' lạ (không có trong bảng Movies): {len(movie_mo_coi)}")

# Kiểm tra xem có cast_id nào trong file liên kết mà KHÔNG tồn tại bên file cast không
cast_mo_coi = df_link[~df_link['cast_id'].isin(df_cast['cast_id'])]
print(f"Số dòng chứa 'cast_id' lạ (không có trong bảng Cast): {len(cast_mo_coi)}")

movies_khong_cast = df_movies[~df_movies['movie_id'].isin(df_link['movie_id'])]
print(f"Số lượng phim bị 'trống' diễn viên: {len(movies_khong_cast)}")

# 2. Tìm các diễn viên KHÔNG THAM GIA bộ phim nào trong bảng liên kết
cast_khong_movie = df_cast[~df_cast['cast_id'].isin(df_link['cast_id'])]
print(f"Số lượng diễn viên không đóng phim nào: {len(cast_khong_movie)}")



Đọc thành công các file dữ liệu!
Kích thước file liên kết gốc: (243885, 3)
Số lượng dòng bị trùng lặp cả cặp (movie_id, cast_id): 0
Số dòng chứa 'movie_id' lạ (không có trong bảng Movies): 0
Số dòng chứa 'cast_id' lạ (không có trong bảng Cast): 0
Số lượng phim bị 'trống' diễn viên: 0
Số lượng diễn viên không đóng phim nào: 0


In [9]:
import pandas as pd

# 1. Đọc file liên kết và các file thực thể (Hãy đổi tên file .csv cho đúng với máy bạn)
try:
    df_link = pd.read_csv('movie_genre.csv')
    df_movies = pd.read_csv('movies.csv')  # Sử dụng file movies đã làm sạch ở bước trước
    df_genre = pd.read_csv('genre.csv')
    print("Đọc thành công các file dữ liệu!")
except Exception as e:
    print(f"Lỗi đọc file: {e}")

# --- BƯỚC 1: KIỂM TRA VÀ XÓA TRÙNG LẶP CẶP KHÓA HỢP THÀNH ---
print(f"Kích thước file liên kết gốc MOVIE_GENRE: {df_link.shape}")

# Kiểm tra trùng lặp dựa trên cả cặp khóa ngoại (theo đúng sơ đồ ERD của bạn)
# Lưu ý: subset dùng đúng tên cột trong file của bạn (ví dụ: 'movie_id' và 'gende_id' hoặc 'genre_id')
trung_lap_cap = df_link[df_link.duplicated(subset=['movie_id', 'genre_id'], keep=False)]
print(f"Số lượng dòng bị trùng lặp cặp (movie_id, gende_id): {len(trung_lap_cap)}")

# Tiến hành xóa trùng lặp, giữ lại dòng đầu tiên
df_link = df_link.drop_duplicates(subset=['movie_id', 'genre_id'], keep='first')
print(f"Kích thước sau khi xóa trùng lặp cặp: {df_link.shape}")

Đọc thành công các file dữ liệu!
Kích thước file liên kết gốc MOVIE_GENRE: (13610, 2)
Số lượng dòng bị trùng lặp cặp (movie_id, gende_id): 0
Kích thước sau khi xóa trùng lặp cặp: (13610, 2)


In [10]:
# --- BƯỚC 2: KIỂM TRA TÍNH TOÀN VẸN THAM CHIẾU (FOREIGN KEY) ---
# Kiểm tra xem có movie_id nào trong file liên kết mà KHÔNG tồn tại bên file movies không
movie_mo_coi = df_link[~df_link['movie_id'].isin(df_movies['movie_id'])]
print(f"Số dòng chứa 'movie_id' lạ (không có trong bảng Movies): {len(movie_mo_coi)}")

# Kiểm tra xem có gende_id nào trong file liên kết mà KHÔNG tồn tại bên file genre không
genre_mo_coi = df_link[~df_link['genre_id'].isin(df_genre['genre_id'])]
print(f"Số dòng chứa 'gende_id' lạ (không có trong bảng Genre): {len(genre_mo_coi)}")

Số dòng chứa 'movie_id' lạ (không có trong bảng Movies): 1012
Số dòng chứa 'gende_id' lạ (không có trong bảng Genre): 0


In [16]:
# --- BƯỚC 3: LỌC BỎ DỮ LIỆU MỒ CÔI VÀ CÔ LẬP ĐỂ SẠCH DATA 100% ---
# 1. Loại bỏ các dòng mồ côi trong bảng liên kết để tránh lỗi Foreign Key trong SQL
df_link = df_link[df_link['movie_id'].isin(df_movies['movie_id'])]
df_link = df_link[df_link['genre_id'].isin(df_genre['genre_id'])]
print(f"Kích thước file liên kết sau khi drop mồ côi: {df_link.shape}")
movies_khong_genre = df_movies[~df_movies['movie_id'].isin(df_link['movie_id'])]
print("Movie không có genre", movies_khong_genre.shape)
# 2. Loại bỏ các thể loại phim không có bất kỳ bộ phim nào thuộc về nó (Dữ liệu cô lập)
df_genre_cleaned = df_genre[df_genre['genre_id'].isin(df_link['genre_id'])]
print(df_genre_cleaned.shape)
# 3. Loại bỏ các bộ phim không có thể loại nào (Nếu tiêu chí hệ thống khuyến nghị yêu cầu)
df_movies_cleaned = df_movies[df_movies['movie_id'].isin(df_link['movie_id'])]
print(df_movies_cleaned.shape)

Kích thước file liên kết sau khi drop mồ côi: (12598, 2)
Movie không có genre (0, 5)
(23, 2)
(4868, 5)


In [17]:
df_link.to_csv("movie_genre_cleaned.csv", index=False)

In [19]:
import pandas as pd

# 1. Đọc file liên kết và các file thực thể (Hãy đổi tên file .csv cho đúng với máy bạn)
try:
    df_link = pd.read_csv('movie_companies.csv')
    df_movies = pd.read_csv('movies.csv')     # Sử dụng file movies sạch nhất từ bước trước
    df_company = pd.read_csv('companies.csv')
    print("Đọc thành công các file dữ liệu!")
except Exception as e:
    print(f"Lỗi đọc file: {e}")

# --- BƯỚC 1: KIỂM TRA VÀ XÓA TRÙNG LẶP CẶP KHÓA HỢP THÀNH ---
print(f"\nKích thước file liên kết gốc COMPANIES_MOVIE: {df_link.shape}")

# Kiểm tra trùng lặp dựa trên cả cặp khóa ngoại (movie_id, company_id)
trung_lap_cap = df_link[df_link.duplicated(subset=['movie_id', 'company_id'], keep=False)]
print(f"Số lượng dòng bị trùng lặp cặp (movie_id, company_id): {len(trung_lap_cap)}")

# Tiến hành xóa trùng lặp, giữ lại dòng đầu tiên
df_link = df_link.drop_duplicates(subset=['movie_id', 'company_id'], keep='first')
print(f"Kích thước sau khi xóa trùng lặp cặp: {df_link.shape}")

Đọc thành công các file dữ liệu!

Kích thước file liên kết gốc COMPANIES_MOVIE: (14932, 2)
Số lượng dòng bị trùng lặp cặp (movie_id, company_id): 0
Kích thước sau khi xóa trùng lặp cặp: (14932, 2)


In [20]:
# --- BƯỚC 2: KIỂM TRA TÍNH TOÀN VẸN THAM CHIẾU (FOREIGN KEY) ---
# Kiểm tra xem có movie_id nào trong file liên kết mà KHÔNG tồn tại bên file movies không
movie_mo_coi = df_link[~df_link['movie_id'].isin(df_movies['movie_id'])]
print(f"Số dòng chứa 'movie_id' lạ (không có trong bảng Movies): {len(movie_mo_coi)}")

# Kiểm tra xem có company_id nào trong file liên kết mà KHÔNG tồn tại bên file company không
company_mo_coi = df_link[~df_link['company_id'].isin(df_company['company_id'])]
print(f"Số dòng chứa 'company_id' lạ (không có trong bảng Company): {len(company_mo_coi)}")

Số dòng chứa 'movie_id' lạ (không có trong bảng Movies): 1149
Số dòng chứa 'company_id' lạ (không có trong bảng Company): 0


In [32]:
# --- BƯỚC 3: KIỂM TRA DỮ LIỆU CÔ LẬP (KHÔNG CÓ TRONG BẢNG LIÊN KẾT) ---
# Tìm các bộ phim KHÔNG CÓ công ty sản xuất nào trong bảng liên kết
df_link = df_link[df_link['movie_id'].isin(df_movies['movie_id'])]
df_link = df_link[df_link['company_id'].isin(df_company['company_id'])]
print(f"Kích thước file liên kết sau khi drop mồ côi: {df_link.shape}")
movies_khong_company = df_movies[~df_movies['movie_id'].isin(df_link['movie_id'])]
print(f"Số lượng phim không có công ty sản xuất nào: {len(movies_khong_company)}")
# Tìm các công ty sản xuất KHÔNG CÓ bộ phim nào trong bảng liên kết
company_khong_movie = df_company[~df_company['company_id'].isin(df_link['company_id'])]
print(f"Số lượng công ty sản xuất trống (không có phim nào): {len(company_khong_movie)}")


Kích thước file liên kết sau khi drop mồ côi: (13783, 2)
Số lượng phim không có công ty sản xuất nào: 348
Số lượng công ty sản xuất trống (không có phim nào): 308


In [33]:
# --- BƯỚC 4: LỌC BỎ DỮ LIỆU LỖI VÀ CÔ LẬP ĐỂ SẠCH DATA 100% ---


# 2. Loại bỏ các công ty sản xuất trống không có phim (Dữ liệu cô lập)
df_company_cleaned = df_company[df_company['company_id'].isin(df_link['company_id'])]

# 3. Loại bỏ các bộ phim không có công ty sản xuất (Đồng bộ với tiêu chí hệ thống khuyến nghị)
df_movies_cleaned = df_movies[df_movies['movie_id'].isin(df_link['movie_id'])]

In [31]:

df_link.to_csv("clean_final/companies_movie_cleaned.csv", index=False)
df_company_cleaned.to_csv("clean_final/company_final.csv", index=False)
df_movies_cleaned.to_csv("movies_final.csv", index=False) # Ghi đè cập nhật lại danh sách phim sạch nhất

print("\n--- HOÀN THÀNH LÀM SẠCH CẶP MOVIE - COMPANY ---")
print(f"Số lượng công ty sản xuất còn lại: {len(df_company_cleaned)}")
print(f"Số lượng phim còn lại sau tất cả các bước lọc: {len(df_movies_cleaned)}")


--- HOÀN THÀNH LÀM SẠCH CẶP MOVIE - COMPANY ---
Số lượng công ty sản xuất còn lại: 5374
Số lượng phim còn lại sau tất cả các bước lọc: 4520


In [34]:
import pandas as pd

# 1. Đọc file liên kết và các file thực thể (Hãy đổi tên file .csv cho đúng với máy bạn)
try:
    df_link = pd.read_csv('movie_keyword.csv')
    df_movies = pd.read_csv('movies.csv')     # Sử dụng file movies sạch nhất từ bước trước
    df_keyword = pd.read_csv('keyword.csv')
    print("Đọc thành công các file dữ liệu!")
except Exception as e:
    print(f"Lỗi đọc file: {e}")

# --- BƯỚC 1: KIỂM TRA VÀ XÓA TRÙNG LẶP CẶP KHÓA HỢP THÀNH ---
print(f"\nKích thước file liên kết gốc MOVIE_KEYWORD: {df_link.shape}")

# Kiểm tra trùng lặp dựa trên cả cặp khóa ngoại (movie_id, keyword_id)
trung_lap_cap = df_link[df_link.duplicated(subset=['movie_id', 'keyword_id'], keep=False)]
print(f"Số lượng dòng bị trùng lặp cặp (movie_id, keyword_id): {len(trung_lap_cap)}")

# Tiến hành xóa trùng lặp, giữ lại dòng đầu tiên
df_link = df_link.drop_duplicates(subset=['movie_id', 'keyword_id'], keep='first')
print(f"Kích thước sau khi xóa trùng lặp cặp: {df_link.shape}")


Đọc thành công các file dữ liệu!

Kích thước file liên kết gốc MOVIE_KEYWORD: (36562, 2)
Số lượng dòng bị trùng lặp cặp (movie_id, keyword_id): 0
Kích thước sau khi xóa trùng lặp cặp: (36562, 2)


In [35]:
# --- BƯỚC 2: KIỂM TRA TÍNH TOÀN VẸN THAM CHIẾU (FOREIGN KEY) ---
# Kiểm tra xem có movie_id nào trong file liên kết mà KHÔNG tồn tại bên file movies không
movie_mo_coi = df_link[~df_link['movie_id'].isin(df_movies['movie_id'])]
print(f"Số dòng chứa 'movie_id' lạ (không có trong bảng Movies): {len(movie_mo_coi)}")

# Kiểm tra xem có keyword_id nào trong file liên kết mà KHÔNG tồn tại bên file keyword không
keyword_mo_coi = df_link[~df_link['keyword_id'].isin(df_keyword['keyword_id'])]
print(f"Số dòng chứa 'keyword_id' lạ (không có trong bảng Keyword): {len(keyword_mo_coi)}")

Số dòng chứa 'movie_id' lạ (không có trong bảng Movies): 2975
Số dòng chứa 'keyword_id' lạ (không có trong bảng Keyword): 0


In [43]:
# --- BƯỚC 3: KIỂM TRA DỮ LIỆU CÔ LẬP (KHÔNG CÓ TRONG BẢNG LIÊN KẾT) ---
# Tìm các bộ phim KHÔNG CÓ từ khóa nào trong bảng liên kết
df_link = df_link[df_link['movie_id'].isin(df_movies['movie_id'])]
df_link = df_link[df_link['keyword_id'].isin(df_keyword['keyword_id'])]
print(f"Kích thước file liên kết sau khi drop mồ côi: {df_link.shape}")


Kích thước file liên kết sau khi drop mồ côi: (33587, 2)


In [44]:
df_link.to_csv("clean_final/movie_keyword_cleaned.csv", index=False)

In [54]:
import pandas as pd

# 1. Đọc file liên kết và các file thực thể (Hãy đổi tên file .csv cho đúng với máy bạn)
try:
    df_link = pd.read_csv('ratings.csv')
    df_movies = pd.read_csv('movies.csv')     # Sử dụng file movies sạch nhất từ bước trước
    df_user = pd.read_csv('users.csv')
    print("Đọc thành công các file dữ liệu!")
except Exception as e:
    print(f"Lỗi đọc file: {e}")

# --- BƯỚC 1: KIỂM TRA VÀ XÓA TRÙNG LẶP CẶP KHÓA HỢP THÀNH ---
print(f"\nKích thước file tương tác gốc RATING: {df_link.shape}")

# Kiểm tra trùng lặp dựa trên cả cặp (user_id, movie_id)
# Một user chỉ được đánh giá một bộ phim một lần duy nhất
trung_lap_cap = df_link[df_link.duplicated(subset=['user_id', 'movie_id'], keep=False)]
print(f"Số lượng dòng bị trùng lặp cặp (user_id, movie_id): {len(trung_lap_cap)}")

# Tiến hành xóa trùng lặp, giữ lại dòng đầu tiên (hoặc dòng cuối tùy bạn chọn)
df_link = df_link.drop_duplicates(subset=['user_id', 'movie_id'], keep='last')
print(f"Kích thước sau khi xóa trùng lặp cặp: {df_link.shape}")


Đọc thành công các file dữ liệu!

Kích thước file tương tác gốc RATING: (49635, 4)
Số lượng dòng bị trùng lặp cặp (user_id, movie_id): 0
Kích thước sau khi xóa trùng lặp cặp: (49635, 4)


In [55]:
# --- BƯỚC 2: KIỂM TRA TÍNH TOÀN VẸN THAM CHIẾU (FOREIGN KEY) ---
# Kiểm tra xem có movie_id nào trong file rating mà KHÔNG tồn tại bên file movies không
movie_mo_coi = df_link[~df_link['movie_id'].isin(df_movies['movie_id'])]
print(f"Số dòng chứa 'movie_id' lạ (không có trong bảng Movies): {len(movie_mo_coi)}")

# Kiểm tra xem có user_id nào trong file rating mà KHÔNG tồn tại bên file user không
user_mo_coi = df_link[~df_link['user_id'].isin(df_user['user_id'])]
print(f"Số dòng chứa 'user_id' lạ (không có trong bảng User): {len(user_mo_coi)}")

Số dòng chứa 'movie_id' lạ (không có trong bảng Movies): 0
Số dòng chứa 'user_id' lạ (không có trong bảng User): 0
